In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("geotech_kmeans_homework.ipynb")

# Geotechnical and Geoenvironmental Engineering: Clustering Compaction Tests with K-means

> **Mohamad M. Hallal, PhD** <br> Teaching Professor, UC Berkeley

[![License](https://img.shields.io/badge/license-CC%20BY--NC--ND%204.0-blue)](https://creativecommons.org/licenses/by-nc-nd/4.0/)
***

In [ ]:
# Please run this cell, and do not modify the contents

import hashlib
def get_hash(num):
    """Helper function for assessing correctness"""
    return hashlib.md5(str(num).encode()).hexdigest()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.vq import whiten, kmeans2, vq

plt.rcParams["figure.figsize"] = (7, 5)
pd.set_option("display.max_columns", 20)

## Background: Proctor Test and Soil-Type Differences

The **Standard Proctor compaction test** (ASTM D698) determines how dry unit weight varies with water content for a given soil. Soil is compacted in a standardized mold at several water contents, and the dry unit weight is measured at each point. The results of this test are used in many areas of geotechnical engineering practice, with the most commonly used attributes from it being the optimal moisture content and maximum dry density.

| Quantity | Symbol | Units | Meaning |
|---|---|---|---|
| **Optimum Moisture Content** | OMC | % | This is the water content where maximum dry unit weight is achieved. Common theory is that water contents below the OMC decrease MDD by introducing more friction, while water contents above the OMC decrease MDD by filling too much of the soil volume with water.|
| **Maximum Dry Density** | MDD | kN/m³ | This is the maximum dry unit weight of a soil under the compaction test. It's value is commonly used to see how dense a soil can be made given a certain method of compaction.|

---

In this assignment, you will use proctor test data to extract the OMC and MDD of many soil samples. You will then explore how these two properties of soils interact with and relate to other soil properties and classifications.

## Part 1: Extracting the OMC and MDD from compaction test data

A regional geotechnical laboratory recently conducted Proctor compaction tests on **36 soil specimens** collected from various sites. Run the cell below to load the CSV file. You do not need to modify this cell.

In the first part of this assignment, we will program a process that can be used to find the OMC and MDD of eeach soil specimen in this data set. 

---

- **`compaction_tests`** — This table holds the data regarding the compaction tests performed on the soil samples. test_id refers to the specific sample of soil while point_id refers to a specific test done to that sample. Looking at the data, you cansee that for soil T01 there were 5 tests done, each giving us a different point on the moisture content vs dry unit weight curve.


In [ ]:
# Run this cell — do not modify this part
compaction_tests   = pd.read_csv("resources/proctor_curves.csv")

# Printing out shape of the compaction_tests table and sample of the table
print("compaction_tests table shape:", compaction_tests.shape)
print("\ncompaction_tests table (first 10 rows):")
display(compaction_tests.head(10))

## Question 1: Fit a Polynomial to a Compaction Curve

There are many ways one could find the omptimum moisture content (OMC) and maximum dry density (MDD) of a soil. A simple way could be to take the sample with the largest dry density and use its mositure content and dry density as the OMC and MDD respectively. This method however is sensitive to measurement noise and experimental error. The test done on the sample at the optimal moisture content is not guarenteed to be measured or done completely accurately. As you may have seen in class, a commonly used method in practice is to fit a parabolic curve through the data points that you get from the compaction test and take the peak of this curve to be the point with coordinates equal to the OMC and MDD. We will mimic this process using a computer program so that it may be easily applied to the whole data set in an objective and accurate manner.

Write a function named `fitProctorCurve` that fits a polynomial of a given degree to a set of compaction measurements and returns the resulting polynomial as a function. We will use this function later to get the OMC and MDD by obtaining its peak.

### Input arguments

| Argument | Type | Description |
|---|---|---|
| `moisture_pcts` | array-like | These are the moisture content values for each compaction point (%) |
| `dry_unit_weights` | array-like | The dry unit weights corresponding to the moisture content values (kN/m³) |
| `degree` | int | The degree of the fitting polynomial (default is `2` for a quadratic shape) |

### Output

| Return value | Type | Description |
|---|---|---|
| `poly_fn` | `numpy.poly1d` | A polynomial function fitted to the compaction data. |

---
We have provided sample code below that creates a figure of the polynomial fitting on a soil sample using the `fitProctorCurve()` function that you create. Use it to visualize and verify your funcion. An example curve fitted to a two degree polynomial is shown below.
<center><img src="resources/Curve_Example.png" width='650'/></center>
    
---
For testing purposes, please use np.polyfit in order to fit your polynomials. If you don't, your code may not pass the autograder. Documentation is linked below. We also encourage querying an AI tool to help with the process if you don't quite understand how the functions work.
- np.polyfit (getting polynmial coefficients): https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html
- np.poly1d (turning coefficients into usable function): nhttps://numpy.org/doc/stable/reference/generated/numpy.poly1d.html

In [ ]:
def fitProctorCurve(moisture_pcts, dry_unit_weights, degree=2):
    # Convert to NumPy arrays for use
    moisture_pcts    = np.array(moisture_pcts)
    dry_unit_weights = np.array(dry_unit_weights)
    
    # Get polynomial coefficients using np.polyfit
    coeffs = ...

    # Create function from coefficients using np.poly1d
    poly_fn = ...

    return poly_fn

In [ ]:
# Helper cell: plots one fitted compaction curve. We made this using an AI tool, and encourage using LLMs to quickly scope out how data looks like by plotting.
example_id = ...

one_curve = compaction_tests[compaction_tests["test_id"] == example_id].sort_values("moisture_content_pct")
w  = one_curve["moisture_content_pct"].to_numpy()
gd = one_curve["dry_unit_weight_kN_m3"].to_numpy()

poly_fn_example = fitProctorCurve(w, gd, degree=2)
w_smooth = np.linspace(w.min(), w.max(), 300)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(w, gd, color="black", zorder=5, s=50, label="Measured points")
ax.plot(w_smooth, poly_fn_example(w_smooth), color="royalblue", linewidth=2,
        label="Degree-2 polynomial fit")
ax.set_xlabel("Moisture Content (%)")
ax.set_ylabel("Dry Unit Weight (kN/m³)")
ax.set_title(f"Proctor compaction curve — Test {example_id}")
ax.legend()
ax.grid(True, linestyle=":", linewidth=0.8)
plt.tight_layout()
plt.show()

#### Testing Cell:

In [ ]:
grader.check("q1")

## Question 2: Extract OMC and MDD from the Fitted Curve

Now that you can fit a polynomial to a compaction curve, you need to find its peak, the point where dry unit weight is maximized. This peak defines the OMC (Optimal Moisture Content) and MDD (Maximum Dry Density).

For a degree-2 polynomial, the peak is a single well-defined maximum that can be found using basic calculus. For higher degree polynomials, a practical and numerically robust approach is to evaluate the polynomial at many closely spaced moisture content values and find which one gives the highest dry unit weight. This sampling method is an example of a procedure called grid search, and is relatively easy in this case since were trying to maximize across a 1D domain which is bounded.

Write a function named `computeOMC_MDD` that returns the OMC and MDD from a fitted polynomial by sampling the polynomial in 1000 equally spaced points from w_min to w_max.

### Input arguments

| Argument | Type | Description |
|---|---|---|
| `poly_fn` | `numpy.poly1d` | This is the polynomial function from `fitProctorCurve`. Calling `poly_fn(x)` outputs the y value of the polynomial curve at point x. (In this case the dry density at inputted moisture content). |
| `w_min` | float | Lower bound of the moisture content range (%) |
| `w_max` | float | Upper bound of the moisture content range (%) |

### Output

| Return value | Type | Description |
|---|---|---|
| `omc` | float | Optimum moisture content (%) |
| `mdd` | float | Maximum dry unit weight (kN/m³) |

### Hints
- Create a dense array of moisture contents with `np.linspace(w_min, w_max, 1000)`.
- Evaluate the polynomial at each point with `poly_fn(w_dense)`.
- Use `np.argmax` to locate the index of the maximum dry unit weight.
- Return `omc` and `mdd` as Python `float` values (cast with `float(...)`).

In [ ]:
def computeOMC_MDD(poly_fn, w_min, w_max):

    # Making array of equally spaced points to call the function on
    moisture_contents  = np.linspace(w_min, w_max, 1000)

    # Use the poly function to get densities at specified moisture contents 
    dry_densities = ...

    # get the index of this dry density array with the max value
    idx_max  = ...

    # Extract the values for optimum moisture content and maximum dry density using this index
    omc = float(moisture_contents[idx_max])                     
    mdd = ...
    return omc, mdd

In [ ]:
# Helper: show fitted curve with OMC and MDD for a single test
ex_id = ...

one = compaction_tests[compaction_tests["test_id"] == ex_id].sort_values("moisture_content_pct")
w   = one["moisture_content_pct"].to_numpy()
gd  = one["dry_unit_weight_kN_m3"].to_numpy()

poly = fitProctorCurve(w, gd, degree=2)
omc, mdd = computeOMC_MDD(poly, w.min(), w.max())

w_smooth = np.linspace(w.min(), w.max(), 300)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(w, gd, color="black", s=20, zorder=5)
ax.plot(w_smooth, poly(w_smooth), color="royalblue")
ax.axvline(omc, color="red",   linestyle="--", label=f"OMC = {omc:.1f}%")
ax.axhline(mdd, color="green", linestyle="--", label=f"MDD = {mdd:.2f} kN/m³")
ax.set_title(f"Test {ex_id}")
ax.set_xlabel("w (%)")
ax.set_ylabel("γd (kN/m³)")
ax.legend()
ax.grid(True, linestyle=":", linewidth=0.6)
plt.tight_layout()
plt.show()

#### Testing Cell:

In [ ]:
grader.check("q2")

## Background: K-means Clustering

**K-means** is an unsupervised machine learning algorithm that partitions a dataset into $k$ groups, or clusters, by grouping datapoints which are geometrically close to each other. It works as follows:

1. **Initialize** $k$ centroids (starting positions in feature space).
2. **Assign** each observation to the nearest centroid using Euclidean distance.
3. **Update** each centroid to the mean of all observations assigned to it.
4. **Repeat** steps 2–3 until assignments no longer change.

The result is $k$ clusters, each defined by a centroid, where observations within a cluster are more similar to each other than to observations in other clusters.

### Key properties

| Property | Description |
|---|---|
| **Unsupervised** | No labels are needed — the algorithm finds structure in the data on its own |
| **Distance-based** | Uses Euclidean distance, so features must be on a comparable scale (This gives a reasoning for why the whitening process is used, which will be described later) |
| **Sensitive to initialization** | Different starting centroids can yield different results |
| **You choose $k$** | The number of clusters must be specified in advance based on prior knowledge or heuristics |

---

A helpful youtube video is linked here https://www.youtube.com/watch?v=4b5d3muPQmA. Feel free to look up more online. There's lots of great visualizations and explanations on the web!

## Part 2: Exploring the OMC and MDD using k-means clustering

In the first part of this assignment, we saw how the OMC and MDD were calculated from a compaction test. In this part of the assigment, we will explore other potential attributes and relations that one can discern from a soil's OMC and MDD using k-means clustering. 

To do this, we will use the the same data set of 36 soil samples, but now use the attributes table. The attributes table gives data on the many characterizing attributes of each soil sample. This includes the OMC, MDD, gravel percentage, fine percentage, plasticity index, and so on.

Run the cell below to load the CSV file. You do not need to modify this cell.

---
- **`attributes`** — This gives a row of features, with each row representing a specific soil sample. The test_id lets you match the features of the soil sample to the compaction test data in the compaction_tests table.


In [ ]:
# Run this cell — do not modify this part
attributes = pd.read_csv("resources/soil_compaction_features.csv")

print("attributes table shape:", attributes.shape)
print("\nAttribute summary (first 5 rows):")
display(attributes.head())

## Building a Feature Matrix for Clustering

We will now begin the process of applying K-means to the data. K-means treats each observation as a point in a multi-dimensional feature space and groups observations by proximity to cluster centroids. For this to work well:

1. **Each row** of the feature matrix must represent one observation (one compaction test).
2. **Each column** must represent one numerical feature.
3. The features must be on a **comparable scale**. Otherwise a variable measured in the tens (e.g. OMC ≈ 10–25%) will dominate a variable measured in the ones (e.g. specific gravity ≈ 2.67), and the clustering will effectively ignore the smaller-scale variable.

### Whitening

SciPy’s `scipy.cluster.vq.whiten` rescales each feature by dividing it by its sample standard deviation, so that every whitened feature has a standard deviation of 1.0. The clustering then treats all features equally in Euclidean distance calculations.

$$\tilde{x}_j = \frac{x_j}{\sigma_j}$$

where $x_j$ is the original feature column and $\sigma_j$ is its sample standard deviation.

---
We have decided to implement the process for you so that you can focus on the procedure and the results. Take a quick look at the code to get an idea of what is being done. Please don't edit the code since the results will be used in future parts

In [ ]:
# This function takes in the features table, along with the columns that we want to use in the table. It drops any rows with missing values, converts it into a Numpy array data structure, 
# and then returns both the raw array and the whitened version to be used in the K-means algorithm.
def buildFeatureMatrix(features_table, feature_cols):

    # Keep only the feature_cols
    df    = features_table[feature_cols]

    # Drop rows with missing values
    df    = df.dropna()

    # Convert to numpy array
    features_numpy   = df.to_numpy()                     
    features_whitened = whiten(features_numpy)                           
    return features_numpy, features_whitened

## Question 3: Cluster the Compaction Tests

Write a function named `clusterSoils` that uses the `kmeans2` to perform k-means clustering on a list of features from the soil attributes table. Essentially, you are choosing the variables that you will perform clustering on by choosing which columns from the attributes table to add to the feature matrix and run k-means on.

### Input arguments

| Argument | Type | Description |
|---|---|---|
| `features` | `numpy.ndarray` | This is an array of features that will be used in the k-means process. Each entry of the array should be a string denoting the column from the attributes table to be used as a feature. |
| `k` | int | The number of clusters to form. |
| `seed` | int | K-means is a randomized process. This is a random seed we set so that the results are reproducible and gradable. The default is set to 0. |

### Output

| Return value | Type | Description |
|---|---|---|
| `centroids_white` | `numpy.ndarray`, shape (k, p) | Cluster centroids in whitened space |
| `labels` | `numpy.ndarray`, shape (n,) | Cluster index (0 to k-1) for each observation |
---
    
We have imported the kmeans2 function and provided the `buildFeatureMatrix` function above to build your feature matrix. Your job is basically putting these two components together to form a k-means clustering. The detailed hints below should help in this process. 

### Hints
- Calling buildFeatureMatrix(attributes, features) will return both the features matrix and the whitened features matrix. We will use the whitened features matrix when running the kmeans.
- Calling `kmeans2(features_whitened, k, minit="++", rng=seed)` wll return the centroids of the clusters with respect to the whitened features, as well as the actual labels of the data points.
---
Note: Adding the term minit = "++" to the kmeans call helps initialize the starting kmeans search correctly. Make sure to do this for both grading and optimization purposes.

In [ ]:
def clusterSoils(features, k, seed=0):
    # Build the feature matrix.
    features_numpy, features_whitened = ...

    # Run kmeans2 to get clusters and labels
    centroids_white, labels = ...
    return centroids_white, labels

In [ ]:
# ── Student inputs ────────────────────────────────────────────────────────────
k                   = ...
clustering_features = ...
plotting_features   = ...
# ─────────────────────────────────────────────────────────────────────────────

# Cluster the soils
centroids_white, labels = clusterSoils(clustering_features, k=k, seed=0)

df_clustered = attributes.dropna(subset=clustering_features).copy()
df_clustered["cluster"] = labels

print("=== Cluster counts ===")
display(df_clustered["cluster"].value_counts().sort_index().rename("n_tests"))

colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]
fig, ax = plt.subplots(figsize=(7, 5))

x_feat, y_feat = plotting_features[0], plotting_features[1]
for cl in range(k):
    mask = df_clustered["cluster"] == cl
    ax.scatter(df_clustered.loc[mask, x_feat],
               df_clustered.loc[mask, y_feat],
               color=colors[cl], label=f"Cluster {cl}", s=55, alpha=0.85)
ax.set_xlabel(x_feat)
ax.set_ylabel(y_feat)
ax.set_title(f"K-means clusters: {x_feat} vs {y_feat}  (k = {k})")
ax.legend()
ax.grid(True, linestyle=":", linewidth=0.8)

plt.tight_layout()
plt.show()

#### Testing Cell:

In [ ]:
grader.check("q3")

## Question 4: Relating OMC and MDD to Soil Type

In geotechnical engineering practice, a soil is classified as a **sand**, **silt**, or **clay** based on its **fines content** and **plasticity index**. Both of these attributes are available in the `attributes` table alongside the OMC and MDD, which gives us an opportunity to explore how compaction behavior relates to soil type.

---

The cell below runs k-means clustering using **OMC and MDD** as the only clustering features, grouping the 36 soil samples purely on the basis of their compaction behavior. It then plots those cluster assignments in a graph of Plasticity Index (%) vs Fines Content (%) so you can visually inspect how well the groups separate.

The goal is to see whether clustering on OMC and MDD alone recovers meaningful soil-type groupings. For example, does a higher MDD tend to correspond to sandier soils?

---

Study the plot output, then complete the dictionary `q4` by mapping each cluster index to the soil type it most closely represents. The assignment may not be perfect near the boundaries between groups, but you should find a clear relationship between the two sets of attributes.

```python
q4 = {
    0: "...",   # replace with "Sandy", "Silty", or "Clayey"
    1: "...",
    2: "...",
}
```

---

Included in the output of the cell is a comparison of the assigned clusterings with the actual assignments of sand, silt, and clay. When we ran the procedure the correctness hovered around **64%**. Not too bad, considering the fact that classification as fine, silt, or clay relies on an arbitrary distinction that may not necessarily match the distribution of the samples that we are given. What's important is the **relation between the attributes**, not necessarily the classification classes — we are just using the classifications of sand, silt, and clay to give something concrete to compare our observed clusterings to.

In [ ]:
# ── Student input: map each cluster index to a soil type ─────────────────────
q4 = {
    ...
    ...
    ...
}
# ─────────────────────────────────────────────────────────────────────────────

# Cluster on OMC and MDD
centroids_white, labels = clusterSoils(["omc_pct", "mdd_kN_m3"], k=3, seed=0)

df_clustered = attributes.dropna(subset=["omc_pct", "mdd_kN_m3"]).copy()
df_clustered["cluster"] = labels
df_clustered["soil_group"] = df_clustered["cluster"].map(q4)

group_colors = {"Sandy": "tab:blue", "Silty": "tab:orange", "Clayey": "tab:green"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: OMC vs MDD — the features used for clustering
for group, grp_df in df_clustered.groupby("soil_group"):
    axes[0].scatter(grp_df["omc_pct"], grp_df["mdd_kN_m3"],
                    color=group_colors.get(group, "gray"), label=group, s=55, alpha=0.85)
axes[0].set_xlabel("OMC (%)")
axes[0].set_ylabel("MDD (kN/m³)")
axes[0].set_title("Predicted soil groups: OMC vs MDD")
axes[0].legend()
axes[0].grid(True, linestyle=":", linewidth=0.8)

# Plot 2: Fines vs Plasticity Index — independent validation view
for group, grp_df in df_clustered.groupby("soil_group"):
    axes[1].scatter(grp_df["fines_pct"], grp_df["plasticity_index_pct"],
                    color=group_colors.get(group, "gray"), label=group, s=55, alpha=0.85)
axes[1].set_xlabel("Fines Content (%)")
axes[1].set_ylabel("Plasticity Index (%)")
axes[1].set_title("Predicted soil groups: Fines vs Plasticity Index")
axes[1].legend()
axes[1].grid(True, linestyle=":", linewidth=0.8)

plt.suptitle("Predicted soil group assignments", fontsize=13)
plt.tight_layout()
plt.show()

# Compare predictions to the answer key
answer_key = pd.read_csv("resources/instructor_answer_key.csv")
true_col = [c for c in answer_key.columns if c != "test_id"][0]

comparison = df_clustered[["test_id", "soil_group"]].merge(answer_key, on="test_id")
comparison = comparison.rename(columns={true_col: "true_group"})
comparison["correct"] = comparison["soil_group"] == comparison["true_group"]
accuracy = comparison["correct"].mean()

print(f"Accuracy: {accuracy:.1%}  ({comparison['correct'].sum()}/{len(comparison)} tests correct)")
display(comparison)

#### Testing Cell:

In [ ]:
grader.check("q4")

## Reflection

Answer the following questions in the markdown cells provided.

### Reflection 1

Re-run the clustering using different variables in feature_cols. Compare the resulting patterns that the algorithm picks up in the data depending on what features you select.

**How does changing the features to run on change the clusterings. Are there any new associations that you can see?**

**Your answer:**

*[double-click to edit]*

### Reflection 2

Re-run `clusterSoils` with `k = 2` and `k = 4`. Describe what you observe in the resulting scatter plots.

**What happens to the silty group when $k = 2$? Does increasing to $k = 4$ produce a geotechnically meaningful fourth group, or does it split an existing group arbitrarily?**

**Your answer:**

*[double-click to edit]*

### Reflection 3

K-means assumes that clusters are roughly **spherical and equal in size** in the feature space, and it uses Euclidean distance as its similarity measure.

**Give a concrete geotechnical scenario where these assumptions might cause K-means to incorrectly group two tests that a geotechnical engineer would classify differently. Use specific feature values in your example.**

**Your answer:**

*[double-click to edit]*

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

Make sure you submit the .zip file to Gradescope.

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)